# 面试问题：Chunked Prefill 怎样降低 decode 干扰，同时避免长 Prompt 饿死？

**回答主线。** 不切分的长 prefill 会占据一次迭代的大量计算，让已在 decode 的请求出现 TPOT 尖峰。Chunked Prefill 把 prompt 拆成 token 块，每轮优先放 decode，再用剩余 token budget 放一个或多个 prefill chunk，使 decode 搭便车并平滑迭代计算量。

调度器不能只写 `min(chunk_size, remaining)`：还要处理 KV 准入、绝对位置、前缀状态、等待公平、取消回滚、TTFT/TPOT 统计和配置搜索。下面用离散事件模拟，不调用推理服务框架。


In [ ]:
import math
from dataclasses import dataclass, field
from collections import deque
import numpy as np

# 请求状态显式区分 prefill 进度、decode 进度和时间戳。
@dataclass
class Request156:
    request_id: str
    prompt_tokens: int
    max_new_tokens: int
    arrival_ms: float = 0.0
    prefilled: int = 0
    generated: int = 0
    first_token_ms: float | None = None
    decode_times: list = field(default_factory=list)
    cancelled: bool = False

r156 = Request156("r", 100, 5)
assert r156.prefilled == 0 and r156.generated == 0
assert r156.first_token_ms is None
assert not r156.cancelled


## 1. 每轮 batch 用统一 token budget 表达

Decode 请求每轮各消费一个新 token；prefill chunk 消费一段 prompt token。调度计划必须记录请求、阶段、绝对起止位置，执行器才能正确追加 KV。


In [ ]:
# WorkItem 用半开区间绑定本轮提交的绝对 token 范围。
@dataclass(frozen=True)
class WorkItem156:
    request_id: str
    phase: str
    start: int
    stop: int

    @property
    def tokens(self):
        return self.stop - self.start

item156 = WorkItem156("r", "prefill", 0, 32)
assert item156.tokens == 32
assert item156.start == 0 and item156.stop == 32
assert item156.phase in {"prefill", "decode"}


## 2. Decode-maximal 策略先保护活跃请求的 TPOT

每个已完成 prefill 且未结束的请求先获得一个 decode slot，剩余预算再分给最老 prefill。`chunk_size` 是单请求上限，不应越过本轮总预算或剩余 prompt。


In [ ]:
def schedule_round156(requests, token_budget, chunk_size):
    # 先按到达时间和 ID 稳定排序 decode，再为 prefill 分配剩余预算。
    plan, remaining = [], token_budget
    decodes = sorted([r for r in requests if not r.cancelled and r.prefilled == r.prompt_tokens and r.generated < r.max_new_tokens], key=lambda r: (r.arrival_ms, r.request_id))
    for request in decodes:
        if remaining < 1:
            break
        plan.append(WorkItem156(request.request_id, "decode", request.generated, request.generated + 1)); remaining -= 1
    prefills = sorted([r for r in requests if not r.cancelled and r.prefilled < r.prompt_tokens], key=lambda r: (r.arrival_ms, r.request_id))
    for request in prefills:
        if remaining <= 0:
            break
        take = min(chunk_size, remaining, request.prompt_tokens - request.prefilled)
        plan.append(WorkItem156(request.request_id, "prefill", request.prefilled, request.prefilled + take)); remaining -= take
    return plan

active156 = Request156("active", 4, 3, prefilled=4)
long156 = Request156("long", 100, 1)
plan156 = schedule_round156([active156, long156], token_budget=16, chunk_size=8)
assert plan156[0].phase == "decode"
assert sum(x.tokens for x in plan156) <= 16
assert any(x.request_id == "long" and x.tokens == 8 for x in plan156)


## 3. 非切分 prefill 会制造 head-of-line blocking

用线性教学成本比较：长 prompt 一次执行时，decode 必须等待整段；切块后每轮只等待一个 chunk。真实 kernel 成本非线性，但尾延迟机制相同。


In [ ]:
def iteration_ms156(prefill_tokens, decode_count, launch_ms=0.1, prefill_ms_token=0.01, decode_ms_token=0.04):
    # 受控模型把两类工作和固定启动开销相加，仅用于比较调度策略。
    return launch_ms + prefill_tokens * prefill_ms_token + decode_count * decode_ms_token

unchunked_delay156 = iteration_ms156(4096, 16)
chunked_delay156 = iteration_ms156(256, 16)
assert unchunked_delay156 > chunked_delay156
assert unchunked_delay156 / chunked_delay156 > 5
assert iteration_ms156(0, 1) > 0


## 4. 离散事件执行器记录 TTFT 与每次 decode 时间

执行 plan 后才能提交 prefill/decode 进度。首个 decode 完成时间定义 TTFT；相邻 decode 完成时间差形成 TPOT。这里把每轮工作视为并行 batch，共享同一个完成时间。


In [ ]:
def execute_plan156(requests, plan, now_ms):
    # 先计算整批成本，再原子提交每个 item，避免半批状态泄漏。
    prefill_tokens = sum(i.tokens for i in plan if i.phase == "prefill")
    decode_count = sum(1 for i in plan if i.phase == "decode")
    finish = now_ms + iteration_ms156(prefill_tokens, decode_count)
    by_id = {r.request_id: r for r in requests}
    for item in plan:
        request = by_id[item.request_id]
        if item.phase == "prefill":
            assert item.start == request.prefilled
            request.prefilled = item.stop
        else:
            assert item.start == request.generated
            request.generated = item.stop
            request.decode_times.append(finish)
            if request.first_token_ms is None:
                request.first_token_ms = finish
    return finish

test_req156 = Request156("t", 8, 2)
now156 = execute_plan156([test_req156], [WorkItem156("t", "prefill", 0, 8)], 0.0)
now156 = execute_plan156([test_req156], [WorkItem156("t", "decode", 0, 1)], now156)
assert test_req156.prefilled == 8 and test_req156.generated == 1
assert test_req156.first_token_ms == now156
assert len(test_req156.decode_times) == 1


## 5. 持续 decode 流量下要为 prefill 保留进度

绝对 decode 优先可能让新请求永远无法完成 prefill。生产策略可设置最小 prefill 配额、最大等待时间或 deadline。下面从总预算中预留 `prefill_reserve`，其余部分仍保护 decode。


In [ ]:
def fair_schedule156(requests, token_budget, chunk_size, prefill_reserve):
    # 有等待 prefill 时限制 decode 占用，为最老 prompt 留出确定预算。
    waiting = any(not r.cancelled and r.prefilled < r.prompt_tokens for r in requests)
    decode_budget = token_budget - min(prefill_reserve, token_budget) if waiting else token_budget
    decode_plan = schedule_round156([r for r in requests if r.prefilled == r.prompt_tokens], decode_budget, chunk_size)
    used = sum(i.tokens for i in decode_plan)
    prefill_plan = schedule_round156([r for r in requests if r.prefilled < r.prompt_tokens], token_budget - used, chunk_size)
    return decode_plan + prefill_plan

many_decode156 = [Request156(f"d{i}", 1, 2, prefilled=1) for i in range(10)]
waiting156 = Request156("waiting", 50, 1)
fair156 = fair_schedule156(many_decode156 + [waiting156], 8, 4, prefill_reserve=4)
assert sum(i.tokens for i in fair156) <= 8
assert any(i.request_id == "waiting" and i.phase == "prefill" for i in fair156)
assert sum(i.phase == "decode" for i in fair156) <= 4


## 6. KV 准入按最终长度预留，取消时可回收

只按当前 chunk 分配可能在 prompt 快完成时才发现 decode 无空间。Admission 应估算 `prompt + max_new_tokens` 的逻辑块上限，并用租户配额/物理水位决定接受；执行中只提交已完成的块。


In [ ]:
class KVAdmission156:
    def __init__(self, capacity_tokens):
        self.capacity_tokens = capacity_tokens
        self.reservations = {}

    def reserve(self, request):
        # request_id 幂等，同一请求重试不会重复占用容量。
        need = request.prompt_tokens + request.max_new_tokens
        if request.request_id in self.reservations:
            return True
        if sum(self.reservations.values()) + need > self.capacity_tokens:
            return False
        self.reservations[request.request_id] = need
        return True

    def release(self, request_id):
        return self.reservations.pop(request_id, 0)

admission156 = KVAdmission156(120)
assert admission156.reserve(Request156("a", 80, 20))
assert not admission156.reserve(Request156("b", 30, 10))
assert admission156.release("a") == 100 and admission156.reserve(Request156("b", 30, 10))


## 7. 分块不能重置绝对位置或前缀状态

用一个简单因果递推验证：一次处理完整序列与按 chunk 处理并携带状态应完全一致。Transformer 中对应的是 KV、position id、RoPE 相位和 block table 的连续提交。


In [ ]:
def causal_recurrence156(tokens, initial=0.0, start_position=0):
    # 位置项依赖全局 start_position，故分块调用必须传递当前位置和状态。
    state, outputs = float(initial), []
    for offset, token in enumerate(tokens):
        position = start_position + offset
        state = 0.7 * state + float(token) + 0.01 * position
        outputs.append(state)
    return np.array(outputs), state

tokens156 = np.arange(1, 21, dtype=np.float64)
full_seq156, full_state156 = causal_recurrence156(tokens156)
first156, state1_156 = causal_recurrence156(tokens156[:7])
second156, state2_156 = causal_recurrence156(tokens156[7:], state1_156, start_position=7)
assert np.allclose(np.concatenate([first156, second156]), full_seq156)
assert math.isclose(state2_156, full_state156)
assert not np.allclose(causal_recurrence156(tokens156[7:], state1_156, 0)[0], second156)


## 8. 配置搜索同时验收 TTFT、TPOT、吞吐与公平

Chunk 越小，decode 干扰通常越低，但启动/调度开销更高，长 prompt TTFT 也可能变差。下面用受控 workload 扫描 chunk，输出 p95 TPOT 代理与完成轮数；生产必须重放真实长度和到达分布。


In [ ]:
def simulate156(chunk_size):
    # 每个配置都新建请求，防止状态从上一次实验泄漏。
    requests = [Request156("long", 128, 4), Request156("short", 8, 6)]
    now, rounds, durations = 0.0, 0, []
    while any(r.generated < r.max_new_tokens for r in requests):
        plan = fair_schedule156(requests, token_budget=32, chunk_size=chunk_size, prefill_reserve=8)
        before = now; now = execute_plan156(requests, plan, now)
        durations.append(now - before); rounds += 1
        if rounds > 100:
            raise RuntimeError("scheduler made no progress")
    return {"chunk": chunk_size, "rounds": rounds, "p95_iteration_ms": float(np.quantile(durations, 0.95)), "finished": all(r.generated == r.max_new_tokens for r in requests)}

sweep156 = [simulate156(c) for c in (4, 8, 16, 32)]
assert all(x["finished"] for x in sweep156)
assert all(x["rounds"] < 100 for x in sweep156)
assert len({x["p95_iteration_ms"] for x in sweep156}) > 1


## 面试总结

- Chunked Prefill 把长 prompt 拆块，在同一迭代优先放 decode，并用剩余预算推进 prefill。
- 它改善 TPOT 干扰但可能增加启动开销和长 prompt TTFT，所以 chunk size 必须基于 workload 搜索。
- 调度正确性包括绝对位置/KV 连续、原子进度提交、KV 最终长度准入、取消回收和 starvation 防护。
- 验收同时报告 TTFT、TPOT、goodput、吞吐、等待公平与按长度 slice 的结果。

延伸阅读：[SARATHI](https://arxiv.org/abs/2308.16369)、[Sarathi-Serve](https://arxiv.org/abs/2403.02310)、[Orca Continuous Batching](https://www.usenix.org/conference/osdi22/presentation/yu)。
